# Santos Exercises

### EX 1

Write down only the basis vectors that have a fixed number of up-spins, Nup. Use L= 6 and Nup = L/2.


VERY IMPORTANT EXERCISE!


We could, of course, use some if-statement to select these states from the total set we generated above. But instead, let us
write a code that generates from the beginning only those specific desired vectors.
In Mathematica, we can type one of these basis vectors and then use the command ‘Permutations[OneBasisVector]’ to get all
the others. In Fortran there is a subroutine called NEXKSB that does a similar job.

In [3]:
# Fixed-Nup basis generator (L=6, Nup = L/2)
from itertools import combinations

L = 6
Nup = L // 2
basis_fixed = []
for pos in combinations(range(L), Nup):
    v = [0] * L
    for i in pos:
        v[i] = 1
    basis_fixed.append(tuple(v))

print(f"L = {L}, Nup = {Nup}, DimFixedUp = {len(basis_fixed)}")
for idx, v in enumerate(basis_fixed):
    print(f"{idx:2d}: {''.join(str(x) for x in v)}")


L = 6, Nup = 3, DimFixedUp = 20
 0: 111000
 1: 110100
 2: 110010
 3: 110001
 4: 101100
 5: 101010
 6: 101001
 7: 100110
 8: 100101
 9: 100011
10: 011100
11: 011010
12: 011001
13: 010110
14: 010101
15: 010011
16: 001110
17: 001101
18: 001011
19: 000111


In [5]:
# Exercise: diagonal elements of H_open_ZZ and H_closed_ZZ in the fixed-Nup basis
# Use L = 6, Nup = L//2 and Jz = 1.0; show only diagonal elements

L = 6
Nup = L // 2
Jz = 1.0

basis_local = [tuple(1 if i in pos else 0 for i in range(L)) for pos in combinations(range(L), Nup)]

def diag_energy_zz(state, closed=False):
    e = 0.0
    for k in range(L-1):
        e += (Jz/4.0) if state[k] == state[k+1] else (-Jz/4.0)
    if closed:
        e += (Jz/4.0) if state[-1] == state[0] else (-Jz/4.0)
    return e

diagonals_open = [diag_energy_zz(s, closed=False) for s in basis_local]
diagonals_closed = [diag_energy_zz(s, closed=True) for s in basis_local]

print('Open chain diagonal elements:')
print(diagonals_open)
print('\nClosed chain diagonal elements:')
print(diagonals_closed)


Open chain diagonal elements:
[0.75, -0.25, -0.25, 0.25, -0.25, -1.25, -0.75, -0.25, -0.75, 0.25, 0.25, -0.75, -0.25, -0.75, -1.25, -0.25, 0.25, -0.25, -0.25, 0.75]

Closed chain diagonal elements:
[0.5, -0.5, -0.5, 0.5, -0.5, -1.5, -0.5, -0.5, -0.5, 0.5, 0.5, -0.5, -0.5, -0.5, -1.5, -0.5, 0.5, -0.5, -0.5, 0.5]


In [ ]:
# Verification: energies from Npair formula vs computed diagonals
# Count Npair for open (L-1 bonds) and closed (L bonds) separately

L = 6
Jz = 1.0

try:
    states = basis_local
    diag_open = diagonals_open
    diag_closed = diagonals_closed
except NameError:
    states = basis_fixed
    def diag_energy_zz(state, closed=False):
        e = 0.0
        for k in range(L-1):
            e += (Jz/4.0) if state[k] == state[k+1] else (-Jz/4.0)
        if closed:
            e += (Jz/4.0) if state[-1] == state[0] else (-Jz/4.0)
        return e
    diag_open = [diag_energy_zz(s, closed=False) for s in states]
    diag_closed = [diag_energy_zz(s, closed=True) for s in states]

# Npair counts
Npair_open = [sum(1 for k in range(L-1) if s[k]==s[k+1]) for s in states]
Npair_closed = [sum(1 for k in range(L) if s[k]==s[(k+1)%L]) for s in states]

E_open_formula = [((2*n - (L-1)) * Jz / 4.0) for n in Npair_open]
E_closed_formula = [((2*n - L) * Jz / 4.0) for n in Npair_closed]

# Compare
matches_open = [abs(a-b) < 1e-12 for a,b in zip(diag_open, E_open_formula)]
matches_closed = [abs(a-b) < 1e-12 for a,b in zip(diag_closed, E_closed_formula)]

print('idx  state   Npair_open  E_open(computed)  E_open(formula)  match')
for i,(s,no,a,b,m) in enumerate(zip(states,Npair_open,diag_open,E_open_formula,matches_open)):
    print(f'{i:2d}: {"".join(map(str,s))}     {no:2d}          {a:6.2f}            {b:6.2f}      {m}')

print('\nAll open matches:', all(matches_open))

print('\nidx  state   Npair_closed  E_closed(computed)  E_closed(formula)  match')
for i,(s,nc,a,b,m) in enumerate(zip(states,Npair_closed,diag_closed,E_closed_formula,matches_closed)):
    print(f'{i:2d}: {"".join(map(str,s))}     {nc:2d}           {a:6.2f}             {b:6.2f}      {m}')

print('\nAll closed matches:', all(matches_closed))


idx  state   Npair  E_open(computed)  E_open(formula)  match
 0: 111000    4       0.75              0.75      True
 1: 110100    2      -0.25             -0.25      True
 2: 110010    2      -0.25             -0.25      True
 3: 110001    3       0.25              0.25      True
 4: 101100    2      -0.25             -0.25      True
 5: 101010    0      -1.25             -1.25      True
 6: 101001    1      -0.75             -0.75      True
 7: 100110    2      -0.25             -0.25      True
 8: 100101    1      -0.75             -0.75      True
 9: 100011    3       0.25              0.25      True
10: 011100    3       0.25              0.25      True
11: 011010    1      -0.75             -0.75      True
12: 011001    2      -0.25             -0.25      True
13: 010110    1      -0.75             -0.75      True
14: 010101    0      -1.25             -1.25      True
15: 010011    2      -0.25             -0.25      True
16: 001110    3       0.25              0.25      True
17: 